### 1. System Initialization & In-Process Connection
By executing `duckdb.connect()`, we instantiate DuckDB as an **in-process** analytical database. Unlike traditional client-server RDBMS (e.g., PostgreSQL or MySQL) that require inter-process communication (IPC) overhead, this lightweight connection allows our application to share the same memory space as the database. This architectural choice enables seamless zero-copy data transfer between Python and DuckDB's **vectorized execution engine**, setting the foundation for high-speed OLAP queries.

In [2]:
%pip install duckdb pandas pyarrow
import duckdb
con = duckdb.connect()

   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   ----------- ---------------------------- 3.7/13.1 MB 20.1 MB/s eta 0:00:01
   ------------------------- -------------- 8.4/13.1 MB 22.4 MB/s eta 0:00:01
   ---------------------------------------  13.1/13.1 MB 22.3 MB/s eta 0:00:01
   ---------------------------------------- 13.1/13.1 MB 21.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


### 2. Data Ingestion & Pipeline Optimization

Here we ingest the raw CO2 emissions dataset to inspect its schema. Notice the `LIMIT 10` clause in our SQL query. In traditional eager-execution engines, the system might parse the entire CSV into memory before applying the limit. DuckDB, however, employs a **pipelined execution model**. 

The `LIMIT` operator is pushed down directly to the CSV scanner. This means DuckDB halts file I/O and parsing immediately after fetching the first 10 rows. This early-termination mechanism avoids unnecessary memory allocation and demonstrates query optimization at the storage-scan layer, even for raw text files.

In [3]:
#load the real dataset
df = con.execute("""
SELECT *
FROM 'data/co-emissions-per-capita.csv'
LIMIT 10
""").df()

df

,Entity,Code,Year,CO₂ emissions per capita
0,Afghanistan,AFG,1949,0.001992
1,Afghanistan,AFG,1950,0.010837
2,Afghanistan,AFG,1951,0.011625
3,Afghanistan,AFG,1952,0.011468
4,Afghanistan,AFG,1953,0.013123
5,Afghanistan,AFG,1954,0.012945
6,Afghanistan,AFG,1955,0.018481
7,Afghanistan,AFG,1956,0.021669
8,Afghanistan,AFG,1957,0.034130
9,Afghanistan,AFG,1958,0.037802


In [4]:
df.columns

Index(['Entity', 'Code', 'Year', 'CO₂ emissions per capita'], dtype='object')

### 3. Application Feature 1: Top Emitters (Filter & Top-K Execution)

This query represents our first core application feature: retrieving the top 10 CO₂ emitters for a specific year. While the SQL logic is straightforward, DuckDB optimizes the physical execution plan significantly. 

After applying the **Vectorized Filter** (`Year = 2024`), the engine encounters an `ORDER BY` paired with a `LIMIT 10`. Instead of performing a full sort on all filtered rows (which is memory-intensive), DuckDB's execution engine dynamically switches to a **Top-N (Top-K) Heap Sort algorithm**. It maintains a bounded priority queue in memory, keeping only the top 10 values and discarding smaller values on the fly. This demonstrates how application-level bounds (`LIMIT`) directly push down into the execution engine to save memory bandwidth and CPU cycles.

In [15]:
con.execute("""
SELECT Entity, Code, Year, "CO₂ emissions per capita" AS co2_pc
FROM 'data/co-emissions-per-capita.csv'
WHERE Year = 2023
ORDER BY co2_pc DESC
LIMIT 10
""").df()

,Entity,Code,Year,co2_pc
0,Qatar,QAT,2023,40.127865
1,Brunei,BRN,2023,27.241413
2,Kuwait,KWT,2023,25.454494
3,Bahrain,BHR,2023,24.710030
4,Trinidad and Tobago,TTO,2023,23.644289
5,Saudi Arabia,SAU,2023,20.365440
6,United Arab Emirates,ARE,2023,19.772913
7,New Caledonia,NCL,2023,17.483173
8,Sint Maarten (Dutch part),SXM,2023,16.156517
9,Oman,OMN,2023,15.806381


### 4. Application Feature 2: Country Trend (Vectorized String Evaluation & Zero-Copy Arrow Integration)

Our second application feature tracks the historical CO₂ emission trend for a specific country. 

Under the hood, the `WHERE Entity = 'United States'` clause triggers DuckDB's **vectorized string evaluation**. Instead of costly row-by-row string matching, DuckDB processes batches of string pointers, minimizing CPU branch mispredictions. 

Furthermore, notice the `.df()` method at the end. Because DuckDB natively operates on the **Apache Arrow** columnar memory format, the result set is transferred to a Pandas DataFrame using **zero-copy integration**. This completely eliminates the costly serialization/deserialization overhead (e.g., JDBC/ODBC overhead) found in traditional client-server databases, allowing our application to render data almost instantaneously.

In [6]:
con.execute("""
SELECT Entity, Year, "CO₂ emissions per capita" AS co2_pc
FROM 'data/co-emissions-per-capita.csv'
WHERE Entity = 'United States'
ORDER BY Year
""").df()

,Entity,Year,co2_pc
0,United States,1800,0.042136
1,United States,1801,0.043749
2,United States,1802,0.046464
3,United States,1803,0.046753
4,United States,1804,0.051548
...,...,...,...
220,United States,2020,13.816895
221,United States,2021,14.758024
222,United States,2022,14.802047
223,United States,2023,14.319450


### 5. Application Feature 3: Country Comparison (Multi-Predicate Pushdown & Vectorized IN Operator)

This query supports our third application feature: cross-comparing specific countries in a given year. 

From an internal perspective, the presence of multiple conditions (`Year = 2024` AND `Entity IN (...)`) triggers DuckDB's **Conjunctive Filter Pushdown**. The query optimizer logically combines these predicates and pushes them directly to the storage layer. This ensures that only the data blocks satisfying *both* conditions are decompressed and loaded into memory.

Furthermore, the `IN` clause is not evaluated as a series of naive `OR` statements. DuckDB's execution engine utilizes a **Vectorized IN Operator**. For a small list of constants like ours, it performs highly optimized SIMD comparisons. If the list were larger, it would dynamically build a perfectly sized hash table in the CPU L1/L2 cache to probe the vector batches, drastically reducing the latency of multi-category filtering.

In [7]:
con.execute("""
SELECT Entity, Year, "CO₂ emissions per capita" AS co2_pc
FROM 'data/co-emissions-per-capita.csv'
WHERE Year = 2024
  AND Entity IN ('United States', 'China', 'India', 'Germany')
ORDER BY co2_pc DESC
""").df()

,Entity,Year,co2_pc
0,United States,2024,14.197287
1,China,2024,8.658390
2,Germany,2024,6.768824
3,India,2024,2.200978


### 6. Storage Layer Transformation: Physical Migration to Parquet

Here, our application physically restructures the dataset from a row-based CSV text file into a **Columnar Storage Format (Parquet)** using DuckDB's native `COPY` pipeline. 

This is not a simple file copy. During execution, DuckDB's engine actively shreds the incoming tuples into **Column Chunks** and organizes them into manageable **Row Groups**. More importantly, the system automatically computes and embeds **Zone Maps (Min-Max metadata)** and applies lightweight compression (e.g., dictionary encoding/RLE) to each column block. 

This physical layout transformation is the absolute prerequisite for our database's optimal performance. It shifts the bottleneck away from disk I/O parsing and unlocks DuckDB's true OLAP power: **Projection Pushdown** (reading only specific columns instead of the entire table) and **Predicate Pushdown** (skipping entire row groups at the storage layer based on query filters).

In [8]:
con.execute("""
COPY 'data/co-emissions-per-capita.csv'
TO 'data/co-emissions-per-capita.parquet'
(FORMAT parquet)
""")

In [9]:
con.execute("""
SELECT *
FROM 'data/co-emissions-per-capita.parquet'
LIMIT 10
""").df()

,Entity,Code,Year,CO₂ emissions per capita
0,Afghanistan,AFG,1949,0.001992
1,Afghanistan,AFG,1950,0.010837
2,Afghanistan,AFG,1951,0.011625
3,Afghanistan,AFG,1952,0.011468
4,Afghanistan,AFG,1953,0.013123
5,Afghanistan,AFG,1954,0.012945
6,Afghanistan,AFG,1955,0.018481
7,Afghanistan,AFG,1956,0.021669
8,Afghanistan,AFG,1957,0.034130
9,Afghanistan,AFG,1958,0.037802


### 7. Application Performance: Parquet vs CSV Execution (The Power of Pushdown)

While the Parquet version returns the exact same top-10 results as the CSV version (confirming logical consistency), the **physical execution path** shifts dramatically under the hood. 

Unlike the CSV execution where the engine was forced to parse the entire text file, querying the Parquet file triggers two massive I/O optimizations:
1. **Projection Pushdown**: DuckDB only reads the required columns (`Entity`, `Code`, `Year`, `CO₂ emissions`) from disk, completely ignoring any other columns.
2. **Predicate Pushdown**: The `WHERE Year = 2024` filter is pushed down into the storage layer. DuckDB reads the Parquet file's Zone Maps (Min-Max indexes) and instantly skips all Row Groups that do not contain the year 2024, without ever loading them into memory. 

This head-to-head comparison perfectly illustrates how matching the right storage format (Columnar) with a Vectorized Execution Engine drastically reduces memory bandwidth and query latency.

In [10]:
con.execute("""
SELECT Entity, Code, Year, "CO₂ emissions per capita" AS co2_pc
FROM 'data/co-emissions-per-capita.parquet'
WHERE Year = 2024
ORDER BY co2_pc DESC
LIMIT 10
""").df()

,Entity,Code,Year,co2_pc
0,Qatar,QAT,2024,41.271180
1,Kuwait,KWT,2024,26.247530
2,Brunei,BRN,2024,26.046202
3,Bahrain,BHR,2024,24.270082
4,Trinidad and Tobago,TTO,2024,22.931944
5,Saudi Arabia,SAU,2024,20.379194
6,United Arab Emirates,ARE,2024,20.131075
7,New Caledonia,NCL,2024,18.064400
8,Sint Maarten (Dutch part),SXM,2024,16.546274
9,Oman,OMN,2024,15.651107


### 8. Deep Dive: Physical Plan Analysis (CSV vs Parquet)

By analyzing the `EXPLAIN` output, we move beyond surface-level benchmarking to verify the **Internal Mapping** between our application logic and DuckDB’s physical execution.

#### **CSV Plan Analysis: The "Eager" Bottleneck**
In the CSV plan, notice the `READ_CSV` operator. Because CSV is a text-based, row-oriented format, the physical plan reveals that DuckDB must initiate a full scan. Although the **Filter** (`Year = 2024`) and **Projection** are applied, they occur *after* the data has been parsed into memory. This confirms our hypothesis: CSV forces the application to pay a "parsing tax" for every row, regardless of the filters used.

#### **Parquet Plan Analysis: The "Lazy" Optimization**
In contrast, the Parquet plan highlights the power of **Metadata-Driven Execution**. The plan reveals two critical optimizations:
* **Filters Pushed Down**: Look for `Filters: Year=2024` inside the `READ_PARQUET` operator itself. This proves the predicate is evaluated at the storage layer using **Zone Maps**, allowing the engine to skip non-matching row groups entirely.
* **Projection Pushed Down**: The plan only lists the 4 required columns in the scan, proving that other data columns never even leave the disk.

#### **Top-K Optimization (Commonality)**
In both plans, the combination of `ORDER BY` and `LIMIT` is fused into a **TOP_N** operator. This confirms that our application behavior (requesting only 10 results) allows DuckDB to avoid a full sort, instead utilizing a memory-efficient min-heap/max-heap to track the top candidates during the streaming scan.

In [11]:
con.execute("""
EXPLAIN
SELECT Entity, Code, Year, "CO₂ emissions per capita" AS co2_pc
FROM 'data/co-emissions-per-capita.parquet'
WHERE Year = 2024
ORDER BY co2_pc DESC
LIMIT 10
""").df()

,explain_key,explain_value
0,physical_plan,┌───────────────────────────┐\n│ ORDE...


In [12]:
con.execute("""
EXPLAIN
SELECT Entity, Code, Year, "CO₂ emissions per capita" AS co2_pc
FROM 'data/co-emissions-per-capita.csv'
WHERE Year = 2024
ORDER BY co2_pc DESC
LIMIT 10
""").df()

,explain_key,explain_value
0,physical_plan,┌───────────────────────────┐\n│ TOP...


#### Physical Plan Strings Output:Result

In [13]:
plan_parquet = con.execute("""
EXPLAIN
SELECT Entity, Code, Year, "CO₂ emissions per capita" AS co2_pc
FROM 'data/co-emissions-per-capita.parquet'
WHERE Year = 2024
ORDER BY co2_pc DESC
LIMIT 10
""").fetchdf()

print(plan_parquet["explain_value"][0])

┌───────────────────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│ "co-emissions-per-capita".│
│ "CO₂ emissions per capita"│
│            DESC           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│           Entity          │
│            Code           │
│            Year           │
│  CO₂ emissions per capita │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: SEMI      │
│                           │
│        Conditions:        │
│  file_index = file_index  ├──────────────┐
│     file_row_number =     │              │
│       file_row_number     │              │
│                           │              │
│          ~0 rows          │              │
└─────────────┬─────────────┘              │
┌─────────────┴─────────────┐┌──────────

In [14]:
plan_csv = con.execute("""
EXPLAIN
SELECT Entity, Code, Year, "CO₂ emissions per capita" AS co2_pc
FROM 'data/co-emissions-per-capita.csv'
WHERE Year = 2024
ORDER BY co2_pc DESC
LIMIT 10
""").fetchdf()

print(plan_csv["explain_value"][0])

┌───────────────────────────┐
│           TOP_N           │
│    ────────────────────   │
│          Top: 10          │
│                           │
│         Order By:         │
│ "co-emissions-per-capita".│
│ "CO₂ emissions per capita"│
│            DESC           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│           Entity          │
│            Code           │
│            Year           │
│           co2_pc          │
│                           │
│        ~7,840 rows        │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│           FILTER          │
│    ────────────────────   │
│       (Year = 2024)       │
│                           │
│        ~7,840 rows        │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       READ_CSV_AUTO       │
│    ────────────────────   │
│         Function:         │
│       READ_CSV_AUTO       │
│                           │
│        P